# Session 1 - The Data Science Workflow & First Contact with Data

**Block 1: Data Science Fundamentals** · 4 hours · Longitudinal project: Barcelona short-term rentals

---

## Learning objectives

By the end of this session you will be able to:

1. Translate a stated business question into a candidate ML formulation, naming the
   **unit of observation**, the **target**, and a **success criterion**. `[CLO1]`
2. Explain the purpose of each stage of the data science workflow, and say what goes
   wrong when a stage is skipped. `[CLO1]`
3. Load an unfamiliar dataset and produce a structured **first-contact report**. `[CLO2, CLO11]`
4. Distinguish questions the data *can* answer from questions it cannot. `[CLO1]`

Each of these is assessed. Objective 3 is your first project milestone (**M0**).

## Prerequisites

Session 0 (asynchronous): Python for data science, NumPy essentials, and a passing
diagnostic. If any of the following makes you hesitate, say so in the first
45 minutes rather than at the end of the session:

- writing a function with a default argument
- the difference between a list and a dictionary
- reading a traceback to find *which line* failed

## Why does this matter?

Most people meet data science backwards. They learn `model.fit(X, y)` first and
spend years wondering why their models disappoint in production.

The order in this course is the order the work actually happens in. Modelling is
**stage 7 of 13**. Six stages come before it, and every one of them can silently
invalidate everything after. This session is about the first two.

## §0 - Setup and reproducibility

Before any analysis: pin the environment and seed the generators. A result you
cannot reproduce is not a result, and "it worked on my machine last Tuesday" is
not a defence you can offer a client.

In [ ]:
import sys
from pathlib import Path

# Make the course library importable from anywhere in the repo.
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data import PATHS, SEED, describe_environment, load_raw, set_seed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print(f"seed = {set_seed(SEED)}")
print(describe_environment().to_string(index=False))

Those version numbers matter more than they look. Between two scikit-learn
releases, the way cross-validation assigns rows to folds changed enough to move
the headline numbers in this course by 0.045. Same data, same code, different
answer. Record your versions.

---

## §1 - What data science actually is

A working definition for this course:

> Data science is the practice of **supporting a decision** using data, and being
> honest about how much support the data actually provides.

Three words are doing the work.

**Decision.** If nothing would be done differently as a result of your analysis,
the analysis has no value, however elegant. Always know whose decision you are
informing.

**Support.** Not "prove". Data constrains belief; it rarely settles a question.

**Honest.** This is the hard one, and it is most of what separates a competent
practitioner from a dangerous one. Your job includes reporting the size of your
own uncertainty, and noticing when a result is too good.

Here is the workflow this course follows. Read the third column carefully - it is
the reason the stages are in this order.

| # | Stage | If you skip it |
|---|---|---|
| 1 | Problem definition | You optimise the wrong quantity, perfectly |
| 2 | Data understanding | You model an artifact of how the data was collected |
| 3 | **Train/test split** | Every number you report afterwards is optimistic |
| 4 | Cleaning | Your model learns your data-entry errors |
| 5 | Exploratory analysis | You miss the structure that dictates your method |
| 6 | Feature engineering | You hand the model raw text and hope |
| 7 | Baseline | You have no idea whether your model is any good |
| 8 | Model | - |
| 9 | Validation | You cannot tell a real gain from noise |
| 10 | Tuning | You leave performance on the table, or overfit the validation set |
| 11 | Evaluation | You report the wrong metric for the decision |
| 12 | Interpretation | You ship something you cannot explain or defend |
| 13 | Communication | The decision-maker ignores you, correctly |

Note where the split sits: **stage 3, before cleaning**. Most tutorials put it
just before modelling. That is the single most consequential ordering error in
applied machine learning, and Session 2 is largely about why.

## §2 - Your two clients

You have one dataset and two clients who want different things from it. This is
normal, and noticing it is part of the job.

### Client A - Direcció de Turisme, Ajuntament de Barcelona

The city has announced it will phase out licensed tourist apartments by 2028.
Enforcement capacity is finite: inspectors can visit a few hundred listings a
month out of more than fifteen thousand. They want to know **which listings are
operating without a valid tourist licence**, so that inspections can be targeted
rather than random.

They also have to defend that targeting in public. If the model concentrates
enforcement in poorer districts, that is a political and legal problem, not
merely a technical one.

### Client B - a property-management company

They manage a growing portfolio and want a **pricing tool**: given a new
apartment's characteristics, what should it be listed at? Their current process
is a manager eyeballing three listings they consider comparable.

### Why two?

Because the same column can be legitimate for one client and disqualifying for
the other, and you cannot tell which without knowing whose decision you serve.
Keep both clients in mind all semester. When you make a modelling choice, the
question "for whom?" usually resolves it.

## §3 - The data

**Inside Airbnb**, Barcelona, snapshot of 24 June 2026.

Inside Airbnb is an activist project that scrapes public Airbnb listing pages to
support housing-policy debate. That origin matters enormously:

- The data was **not collected for your purpose**.
- There is **no documentation contract**. Column names are descriptive, not
  authoritative.
- It contains **artifacts of how the scraper works**, which you will have to find.

This is what real data is like. A dataset assembled for your question, with
documented semantics, is a luxury you should not expect.

> **One rule, starting now:** do not re-download the data. The snapshot is pinned
> in the repository. Inside Airbnb rotates snapshots quarterly, and a fresh
> download would silently change every number you and your classmates compute.

## §4 - First contact

The goal of first contact is **not** to understand the data. It is to find out how
much you do not understand, and to write that down.

Four questions, in order:

1. How big is it, and what is one row?
2. What types are the columns, and do those types make sense?
3. What is missing, and is the missingness patterned?
4. Which columns are unusable, and why?

In [ ]:
df = load_raw()
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

### Predict before you run

Before executing the next cell, **write down your answers** in the cell below it.
Guessing and being wrong is how you find out what your intuitions are worth -
which is the point of the exercise, so do not skip the writing part.

1. One row of this table represents what, exactly?
2. Of 90 columns, how many do you expect to be **completely unusable**?
3. Which single column do you expect to have the most missing values?

In [ ]:
# TODO: Write your three predictions here BEFORE running the next cell.
#
# 1. One row represents:
# 2. Number of completely unusable columns (a number):
# 3. Column with the most missing values:

In [ ]:
# Question 1: what is one row?
print("rows:", f"{len(df):,}")
print("unique listing ids:", f"{df['id'].nunique():,}")
print("unique hosts:      ", f"{df['host_id'].nunique():,}")
print()
print("listings per host, top 5:")
print(df["host_id"].value_counts().head(5).to_string())

Stop and read that output.

15,293 rows, 15,293 unique listing ids - so one row is one listing. But only
**4,595 unique hosts**, and the largest owner controls **588 listings**.

So the rows are *not* independent of each other. That single observation will
dictate how you split your data in Session 2 and how you cross-validate in
Session 6. Write it in your Fact Sheet.

In [ ]:
# Question 2: types and completeness, all 90 columns at once.
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(dropna=True),
})
audit.sort_values("missing_pct", ascending=False).head(20)

### §4.1 - Unusable columns

A column with 100% missing values carries no information. It is not a small
problem to be imputed; it is not a column.

In [ ]:
empty_cols = [c for c in df.columns if df[c].isna().all()]
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

print(f"{len(empty_cols)} columns are 100% empty:")
for c in empty_cols:
    print(f"    {c}")
print(f"\n{len(constant_cols)} columns are constant: {constant_cols}")
print(f"\nUsable columns: {df.shape[1] - len(set(empty_cols) | set(constant_cols))} of {df.shape[1]}")

## §5 - Guided exercise: the client's question

Client B (the property manager) sends you an email:

> *"Can you tell us whether instant-booking listings command a price premium? We're
> considering turning it on across the portfolio and would like the evidence
> first."*

This is a perfectly reasonable business question. Answer it.

In [ ]:
# TODO: Investigate whether you can answer Client B's question.
#
# Steps:
#   1. Find the column that would let you answer it.
#   2. Check whether it is usable.
#   3. Write the reply you would actually send. Two or three sentences.
#
# Hint: start by searching the column names.

### What just happened

You were asked a sensible question about a column that exists by name and is
entirely empty. No modelling technique recovers from this. It is a **stage 2**
finding - data understanding - and if you had skipped stage 2 and gone straight to
modelling, you would have discovered it as a confusing error message somewhere
around stage 8, or worse, not at all.

Two habits to take from this:

1. **A column name is a claim, not a fact.** Verify before you rely on it.
2. **"We cannot answer this" is a legitimate professional deliverable**, provided
   you say why and propose what would change the answer. Clients are far more
   annoyed by a confident wrong answer than by a well-explained "not with this
   data".

## §6 - Independent exercise: the Dataset Fact Sheet (Milestone M0)

This is your first project deliverable. It is formative - it is not graded - but
it is the foundation for M1, and the questions in it recur all semester.

Produce a fact sheet covering:

**A. Shape and grain**
- rows, columns, and what one row is
- how many independent units there really are, and why that differs from the row count

**B. Completeness**
- how many columns are unusable, and the list of them
- the three columns with the most missing values among those that are *not* fully empty

**C. Types**
- at least two columns whose stored type is wrong for their content, with evidence

**D. Questions**
- three questions this data could answer
- one question it cannot, with the reason

**E. Problem statement** - one paragraph naming:
- the client you are serving
- the unit of observation
- the target variable
- a success criterion that a non-technical person could check

In [ ]:
# TODO: Section A - shape and grain.
# Print the shape, then work out how many genuinely independent units there are.

In [ ]:
# TODO: Section B - completeness.
# List the unusable columns, then the three worst *partially* missing columns.

In [ ]:
# TODO: Section C - types.
# Find at least two columns whose dtype does not match their content.
# Print a few example values as evidence for each.

## §7 - Debate: regression or classification?

Twenty minutes, in pairs, then a whole-group round.

You have two clients (§2). For **each** one, decide and defend:

1. Is their question a regression problem or a classification problem?
2. What exactly is the target variable? Name the column, or describe how you would
   construct it.
3. What is the unit of observation?
4. What would a **useful** answer look like - not a good score, but a result that
   changes what the client does?

Then the harder question, which is the real point of the exercise:

> **Is there a single column in this dataset that would be perfectly acceptable to
> use for one client and unacceptable for the other?**

You do not have enough information yet to answer that confidently. Argue about it
anyway, and write down your position. We will return to it in Session 6, and you
will find out whether you were right.

## §8 - Common mistakes in this session

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Treating each row as an independent observation | It is one row per listing, so it feels like one observation | Check the grouping column. Here, 80% of listings share a host with another listing. |
| Trusting a column because its name is descriptive | Names are usually reliable | Check `missing_pct` and `n_unique` before using any column |
| Reporting "the data is clean" | `head()` looked fine | `head()` shows 5 of 15,293 rows. Aggregate before you conclude. |
| Defining success as a metric value | It sounds rigorous | State success in the client's terms; derive the metric from that |
| Starting with the model | It is the part you came here for | Stage 7 of 13 |

## §9 - Reflection

Write two or three sentences on each. These are not busywork; the answers are the
raw material for your final report.

1. You found that a column the client asked about is unusable. What else in this
   dataset might be quietly unusable in a way that `missing_pct` would **not**
   reveal?
2. Your two clients could reach opposite conclusions from the same analysis. Whose
   interests are represented in a dataset scraped by a housing-policy activist
   project, and how might that shape what is in it?
3. What is the single thing you most want to check next, and why that one?

## §10 - Knowledge check

Answer without scrolling back.

1. One row of this dataset is one ______, and there are ______ of them.
2. There are 4,595 hosts and 15,293 listings. Why does that gap matter for a
   train/test split?
3. A column has `dtype=object` and 0% missing values. Name two distinct reasons it
   might still be unusable as a model input.
4. At which stage of the workflow does the train/test split belong, and what does
   placing it later cost you?
5. Client A wants to target inspections. Give one success criterion that is *not*
   a model metric.

## Summary

- Data science supports a **decision**; know whose, and be honest about how much
  support you actually have.
- Modelling is **stage 7 of 13**. The train/test split is **stage 3**, before
  cleaning - and Session 2 is about why that ordering is not negotiable.
- One row here is one **listing**, but 15,293 listings come from only **4,595
  hosts**, and one host owns 588. The rows are not independent, and that fact will
  shape your splitting and validation for the rest of the course.
- **12 of 90 columns are entirely empty**, including one a client explicitly asked
  about. A column name is a claim, not a fact.
- "We cannot answer that with this data" is a professional deliverable when it comes
  with a reason and a proposal.

## Key takeaways

1. Check the grain and the grouping before anything else.
2. Aggregate before you conclude. `head()` shows you 0.03% of this dataset.
3. Define success in the client's language first, then choose a metric to match.

## Further exploration

**Essential**
- James, Witten, Hastie & Tibshirani, *An Introduction to Statistical Learning*,
  ch. 1–2. Free PDF: https://www.statlearning.com/
- pandas user guide, "Essential basic functionality":
  https://pandas.pydata.org/docs/user_guide/basics.html

**Recommended**
- Inside Airbnb, "Get the Data" and the project's stated methodology:
  https://insideairbnb.com/get-the-data/ - read what the project says it is *for*,
  then reread your answer to Reflection question 2.

**Advanced**
- Gebru et al. (2021), *Datasheets for Datasets*, Communications of the ACM.
  https://arxiv.org/abs/1803.09010 - your Fact Sheet is a small datasheet; this is
  the argument for why they should be standard practice.

---

**Next session:** you will split this data before touching it, and then find out
what the cleaning decisions actually cost. Bring your Fact Sheet.